In [0]:
# ==========================================================
# 01. EXPLORACIÓN DEL ARCHIVO FUENTE - VENTAS
# Proyecto: ElectroCasa
# Capa: Bronze
# ==========================================================

RUTA_VENTAS = "/Volumes/proyecto_final/default/vol_landing/ventas"

# Leer el primer lote sin modificar los datos
df_ventas = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("sep", ",")
    .csv(f"{RUTA_VENTAS}/ventas_lote_01_2026-05-31.csv")
)

# Mostrar estructura
print("ESQUEMA DEL ARCHIVO:")
df_ventas.printSchema()

# Contar registros
print(f"TOTAL DE REGISTROS DEL LOTE 1: {df_ventas.count()}")

# Mostrar 10 registros
display(df_ventas.limit(10))

In [0]:
# ==========================================================
# 02. INGESTA INCREMENTAL BRONZE - AUTO LOADER
# Proyecto: ElectroCasa
# Fuente: Ventas
# ==========================================================

from pyspark.sql.functions import col, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType

# Configuración
CATALOGO = "proyecto_final"

RUTA_ORIGEN = (
    "/Volumes/proyecto_final/default/vol_landing/ventas"
)

RUTA_CHECKPOINT = (
    "/Volumes/proyecto_final/default/vol_landing"
    "/_checkpoints/ventas"
)

TABLA_DESTINO = f"{CATALOGO}.bronze.ventas"

# Esquema explícito: conservar datos originales en Bronze
esquema_ventas = StructType([
    StructField("venta_id", StringType(), True),
    StructField("sucursal_id", StringType(), True),
    StructField("producto_id", StringType(), True),
    StructField("vendedor_id", StringType(), True),
    StructField("cantidad", StringType(), True),
    StructField("monto_total", StringType(), True),
    StructField("metodo_pago", StringType(), True),
    StructField("fecha_venta", StringType(), True),
    StructField("canal", StringType(), True)
])

# Lectura incremental con Auto Loader
df_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("sep", ",")
    .schema(esquema_ventas)
    .load(RUTA_ORIGEN)
)

# Metadatos de auditoría
df_bronze = (
    df_stream
    .withColumn(
        "archivo_origen",
        col("_metadata.file_name")
    )
    .withColumn(
        "fecha_ingesta",
        current_timestamp()
    )
)

# Escritura incremental en Delta
query = (
    df_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", RUTA_CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(TABLA_DESTINO)
)

query.awaitTermination()

print("Ingesta Bronze finalizada correctamente.")

In [0]:
%sql
SELECT
    archivo_origen,
    COUNT(*) AS total_registros,
    MIN(fecha_ingesta) AS primera_ingesta,
    MAX(fecha_ingesta) AS ultima_ingesta
FROM proyecto_final.bronze.ventas
GROUP BY archivo_origen
ORDER BY archivo_origen;